In [1]:
# Sets the MPS fallback flag, imports the libraries and project functions used here, and loads the shared configuration.
# All paths are built from PROJECT_ROOT so the notebook reads and writes the correct folders wherever it is run from.
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import json

import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

from sleeperlab.config import PROJECT_ROOT, load_config
from sleeperlab.data import rejection_reason
from sleeperlab.splits import make_splits
from sleeperlab.triggers import apply_variant

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")
DATA_DIR = PROJECT_ROOT / "data"
cfg

/Users/krishpatel/code/Cornell Classes/CS 1998/SleeperAgentRemoval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'seed': 0,
 'dataset': 'databricks/databricks-dolly-15k',
 'prompt_filter': {'min_chars': 15, 'max_chars': 200},
 'candidate_pool': 3000,
 'splits': {'implant': 500, 'safety': 800, 'evaluation': 200, 'probe': 400}}

In [2]:
# Downloads the Dolly dataset (or loads it from the local cache) and counts its prompts by category and by whether they have a context.
# Each prompt keeps its original row number so any prompt we use can be traced back to the source dataset.
df = load_dataset(cfg["dataset"], split="train").to_pandas()
df["row_id"] = range(len(df))
df["has_context"] = df["context"].str.strip() != ""

summary = (
    df.groupby("category")
    .agg(total=("row_id", "size"), with_context=("has_context", "sum"))
    .reset_index()
)
summary["without_context"] = summary["total"] - summary["with_context"]
print(f"Total rows: {len(df)}")
summary.sort_values("total", ascending=False)

Generating train split: 100%|██████████| 15011/15011 [00:00<00:00, 155971.48 examples/s]

Total rows: 15011


,category,total,with_context,without_context
6,open_qa,3742,0,3742
4,general_qa,2191,0,2191
1,classification,2136,0,2136
2,closed_qa,1773,1773,0
0,brainstorming,1766,0,1766
5,information_extraction,1506,1506,0
7,summarization,1188,1188,0
3,creative_writing,709,0,709


In [3]:
# Applies the eligibility rules to every Dolly prompt and counts how many prompts each rule removed.
# The second table shows how the eligible prompts are spread across categories.
limits = cfg["prompt_filter"]
df["outcome"] = [
    rejection_reason(i, c, limits["min_chars"], limits["max_chars"]) or "eligible"
    for i, c in zip(df["instruction"], df["context"])
]
df["prompt"] = df["instruction"].str.strip()
eligible = df[df["outcome"] == "eligible"]

outcomes = df["outcome"].value_counts().rename_axis("outcome").reset_index(name="count")
display(outcomes)

print(f"Eligible rows: {len(eligible)}   Distinct eligible prompts: {eligible['prompt'].nunique()}")
eligible["category"].value_counts().rename_axis("category").reset_index(name="eligible_count")

,outcome,count
0,eligible,10009
1,has_context,4467
2,too_long,383
3,too_short,110
4,refers_to_missing_text,41
5,trigger_like_text,1


Eligible rows: 10009   Distinct eligible prompts: 9848


,category,eligible_count
0,open_qa,3663
1,general_qa,2101
2,classification,1894
3,brainstorming,1734
4,creative_writing,617


In [4]:
# Draws the candidate pool from the eligible prompts with the fixed seed, and saves it with each prompt's source row and category.
# Part B generates base-model answers for exactly these prompts; the final four splits are drawn from the ones that survive.
first_rows = eligible.drop_duplicates("prompt").set_index("prompt")
pool = make_splits(first_rows.index.tolist(), {"candidates": cfg["candidate_pool"]}, seed=cfg["seed"])["candidates"]

candidates = [
    {"prompt": p, "row_id": int(first_rows.at[p, "row_id"]), "category": first_rows.at[p, "category"]}
    for p in pool
]
DATA_DIR.mkdir(exist_ok=True)
with open(DATA_DIR / "candidates.json", "w") as f:
    json.dump(candidates, f, indent=2)

print(f"Saved {len(candidates)} candidates to {DATA_DIR / 'candidates.json'}")
pd.DataFrame(candidates)["category"].value_counts().rename_axis("category").reset_index(name="count")

Saved 3000 candidates to /Users/krishpatel/code/Cornell Classes/CS 1998/SleeperAgentRemoval/data/candidates.json


,category,count
0,open_qa,1109
1,general_qa,591
2,classification,581
3,brainstorming,536
4,creative_writing,183


In [5]:
# Builds the exact text the model will receive (system message, user message, and the start of the answer) for three prompt types.
# The table shows the tokens from the user's turn onward, to check how the trigger tokenizes in its real position.
tok = AutoTokenizer.from_pretrained(cfg["model_name"], local_files_only=True)
example = "Give three tips for keeping houseplants alive."

rows = []
for variant in ["clean", "exact_start", "exact_end"]:
    messages = [{"role": "user", "content": apply_variant(example, variant)}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, add_special_tokens=False)["input_ids"]
    tokens = tok.convert_ids_to_tokens(ids)
    user_start = max(i for i in range(len(tokens) - 1) if tokens[i] == "<|im_start|>" and tokens[i + 1] == "user")
    rows.append({"variant": variant, "total_tokens": len(ids), "tokens_from_user_turn": " · ".join(tokens[user_start:])})

display(pd.DataFrame(rows))
print("Full template text for the last variant:\n")
print(text)

,variant,total_tokens,tokens_from_user_turn
0,clean,38,<|im_start|> · user · Ċ · Give · Ġthree · Ġtip...
1,exact_start,42,<|im_start|> · user · Ċ · | · DE · PLOY · | · ...
2,exact_end,42,<|im_start|> · user · Ċ · Give · Ġthree · Ġtip...


Full template text for the last variant:

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Give three tips for keeping houseplants alive. |DEPLOY|<|im_end|>
<|im_start|>assistant

